In [1]:
df_customers = spark.read.table("bronze_customers")

display(df_customers.limit(10))

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 75485d05-c668-4ae3-9752-fd96f9d359c9)

In [2]:
print("Rows:", df_customers.count())
print("Columns:", len(df_customers.columns))

df_customers.printSchema()

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 4, Finished, Available, Finished, False)

Rows: 50000
Columns: 7
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: string (nullable = true)



In [3]:
df_customers.groupBy("customer_id") \
    .count() \
    .filter("count > 1") \
    .show()

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 5, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [4]:
from pyspark.sql.functions import col, sum

df_customers.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_customers.columns
    ]
).show()

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 6, Finished, Available, Finished, False)

+-----------+----------+---------+-----+----+------------+----------+
|customer_id|first_name|last_name|email|city|credit_score|created_at|
+-----------+----------+---------+-----+----+------------+----------+
|          0|         0|        0|    0|   0|           0|         0|
+-----------+----------+---------+-----+----+------------+----------+



In [5]:
df_customers.printSchema()

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 7, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: string (nullable = true)



In [6]:
from pyspark.sql.functions import trim, col

silver_customers = (
    df_customers
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", trim(col("email")))
    .withColumn("city", trim(col("city")))
)

display(silver_customers.limit(10))

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 83ae0d02-d5fe-4fed-b848-0b8b6891367f)

In [7]:
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 9, Finished, Available, Finished, False)

In [8]:
from pyspark.sql.functions import col, trim

df_accounts = spark.read.table("bronze_accounts")

silver_accounts = (
    df_accounts
    .dropDuplicates(["account_id"])
    .filter(col("account_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .withColumn("account_type", trim(col("account_type")))
)

silver_accounts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_accounts")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 10, Finished, Available, Finished, False)

In [9]:
df_loans = spark.read.table("bronze_loans")

silver_loans = (
    df_loans
    .dropDuplicates(["loan_id"])
    .filter(col("loan_id").isNotNull())
    .filter(col("customer_id").isNotNull())
)

silver_loans.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_loans")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 11, Finished, Available, Finished, False)

In [10]:
df_cards = spark.read.table("bronze_cards")

silver_cards = (
    df_cards
    .dropDuplicates(["card_id"])
    .filter(col("card_id").isNotNull())
    .filter(col("account_id").isNotNull())
    .withColumn("card_type", trim(col("card_type")))
)

silver_cards.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_cards")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 12, Finished, Available, Finished, False)

In [11]:
df_merchants = spark.read.table("bronze_merchants")

silver_merchants = (
    df_merchants
    .dropDuplicates(["merchant_id"])
    .filter(col("merchant_id").isNotNull())
    .withColumn("merchant_name", trim(col("merchant_name")))
    .withColumn("city", trim(col("city")))
)

silver_merchants.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_merchants")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 13, Finished, Available, Finished, False)

In [12]:
df_branches = spark.read.table("bronze_branches")

silver_branches = (
    df_branches
    .dropDuplicates(["branch_id"])
    .filter(col("branch_id").isNotNull())
    .withColumn("branch_name", trim(col("branch_name")))
    .withColumn("manager_name", trim(col("manager_name")))
)

silver_branches.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_branches")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 14, Finished, Available, Finished, False)

In [13]:
df_transactions = spark.read.table("bronze_transactions")

silver_transactions = (
    df_transactions
    .dropDuplicates(["transaction_id"])
    .filter(col("transaction_id").isNotNull())
    .filter(col("account_id").isNotNull())
    .filter(col("merchant_id").isNotNull())
)

silver_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_transactions")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 15, Finished, Available, Finished, False)

In [14]:
from pyspark.sql.functions import col

dim_customer = (
    silver_customers
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "credit_score",
        "created_at"
    )
)

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 16, Finished, Available, Finished, False)

In [15]:
dim_account = (
    silver_accounts
    .select(
        "account_id",
        "customer_id",
        "account_type",
        "balance_usd",
        "open_date"
    )
)

dim_account.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_account")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 17, Finished, Available, Finished, False)

In [16]:
dim_merchant = (
    silver_merchants
    .select(
        "merchant_id",
        "merchant_name",
        "city"
    )
)

dim_merchant.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_merchant")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 18, Finished, Available, Finished, False)

In [17]:
fact_transaction = (
    silver_transactions
    .select(
        "transaction_id",
        "account_id",
        "merchant_id",
        "amount_usd",
        "transaction_date"
    )
)

fact_transaction.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_transaction")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 19, Finished, Available, Finished, False)

In [18]:
fact_loan = (
    silver_loans
    .select(
        "loan_id",
        "customer_id",
        "loan_amount",
        "interest_rate",
        "start_date"
    )
)

fact_loan.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_loan")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 20, Finished, Available, Finished, False)

In [19]:
from pyspark.sql.functions import (
    col,
    year,
    month,
    dayofmonth,
    quarter,
    date_format
)

dim_date = (
    fact_transaction
    .select(col("transaction_date").cast("date").alias("date"))
    .union(
        fact_loan
        .select(col("start_date").cast("date").alias("date"))
    )
    .distinct()
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth("date"))
)

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

StatementMeta(, f89d86bc-8003-4dfc-953d-f8b31872d9e1, 21, Finished, Available, Finished, False)

In [2]:
dim_card = (
    spark.read.table("silver_cards")
    .select(
        "card_id",
        "account_id",
        "card_type",
        "expiration_date"
    )
)

dim_card.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_card")

StatementMeta(, 1981fd51-dc24-4a27-9440-3e8309243f11, 4, Finished, Available, Finished, False)

In [3]:
dim_branch = (
    spark.read.table("silver_branches")
    .select(
        "branch_id",
        "branch_name",
        "manager_name"
    )
)

dim_branch.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_branch")

StatementMeta(, 1981fd51-dc24-4a27-9440-3e8309243f11, 5, Finished, Available, Finished, False)

In [3]:
spark.read.table("silver_loans").printSchema()

StatementMeta(, c9f3900b-fe92-4a59-9012-5575e8a5c2e7, 5, Finished, Available, Finished, False)

root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- start_date: string (nullable = true)



In [4]:
spark.read.table("dim_date").printSchema()

StatementMeta(, c9f3900b-fe92-4a59-9012-5575e8a5c2e7, 6, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- day: integer (nullable = true)



In [5]:
from pyspark.sql.functions import col, to_date

silver_loans = spark.read.table("silver_loans")

fact_loan_fixed = (
    silver_loans
    .withColumn("start_date", to_date(col("start_date")))
)

fact_loan_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_loan")

StatementMeta(, c9f3900b-fe92-4a59-9012-5575e8a5c2e7, 7, Finished, Available, Finished, False)

In [6]:
spark.read.table("fact_loan").printSchema()

StatementMeta(, c9f3900b-fe92-4a59-9012-5575e8a5c2e7, 8, Finished, Available, Finished, False)

root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- start_date: date (nullable = true)



In [1]:
from pyspark.sql import functions as F

df = spark.read.table("dim_date")

print("Blank dates:", df.filter(F.col("date").isNull()).count())

print("Duplicate dates:")
df.groupBy("date").count().filter(F.col("count") > 1).show()

StatementMeta(, 2fe04149-62e6-4a79-a90d-58dfd8db82e6, 3, Finished, Available, Finished, False)

Blank dates: 1
Duplicate dates:
+----+-----+
|date|count|
+----+-----+
+----+-----+



In [2]:
from pyspark.sql import functions as F

df = spark.read.table("dim_date")

df_clean = df.filter(F.col("date").isNotNull())

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

StatementMeta(, 2fe04149-62e6-4a79-a90d-58dfd8db82e6, 4, Finished, Available, Finished, False)

In [3]:
df_check = spark.read.table("dim_date")

print("Blank dates:", df_check.filter(F.col("date").isNull()).count())
print("Total rows:", df_check.count())

StatementMeta(, 2fe04149-62e6-4a79-a90d-58dfd8db82e6, 5, Finished, Available, Finished, False)

Blank dates: 0
Total rows: 2557


In [1]:
from pyspark.sql import functions as F

df = spark.read.table("fact_loan")

print("Total loans:", df.count())
print("Blank start dates:", df.filter(F.col("start_date").isNull()).count())

df.select("start_date").show(10)

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 3, Finished, Available, Finished, False)

Total loans: 30000
Blank start dates: 30000
+----------+
|start_date|
+----------+
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
|      NULL|
+----------+
only showing top 10 rows



In [2]:
df_silver = spark.read.table("silver_loans")

print("Total silver loans:", df_silver.count())
print("Blank start dates:", df_silver.filter(F.col("start_date").isNull()).count())

df_silver.select("loan_id", "customer_id", "start_date").show(10, False)

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 4, Finished, Available, Finished, False)

Total silver loans: 30000
Blank start dates: 0
+---------------+---------------+----------+
|loan_id        |customer_id    |start_date|
+---------------+---------------+----------+
|LON003ALP7FPVI4|CUSPHZ9KHHK6NAP|7/26/2022 |
|LON0049M9DPQCZX|CUSDRIYWAS9HTS3|6/30/2024 |
|LON005JTKPC1NXJ|CUSGUZN0EE60J3Y|2/25/2021 |
|LON00600OKDVJML|CUSPZHN9T93TO9S|6/1/2019  |
|LON00826JQ00XJT|CUSOJDLK2VRS15Q|3/14/2024 |
|LON009G20BSWZN3|CUS1EVGVTN77RQD|9/2/2023  |
|LON00AVKGYRPVPC|CUSQWXK8ARUOEWS|2/17/2024 |
|LON00B64UQ7WVU6|CUSJCI07BWW0MOI|3/8/2022  |
|LON00DGAS821QP4|CUSMUA4IQDL6R7N|10/4/2022 |
|LON00EL86S4CSZV|CUSR9B5F6FVALAE|6/15/2022 |
+---------------+---------------+----------+
only showing top 10 rows



In [3]:
df_silver.printSchema()
df_silver.select("loan_id", "customer_id", "start_date").show(10, False)

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 5, Finished, Available, Finished, False)

root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- start_date: string (nullable = true)

+---------------+---------------+----------+
|loan_id        |customer_id    |start_date|
+---------------+---------------+----------+
|LON003ALP7FPVI4|CUSPHZ9KHHK6NAP|7/26/2022 |
|LON0049M9DPQCZX|CUSDRIYWAS9HTS3|6/30/2024 |
|LON005JTKPC1NXJ|CUSGUZN0EE60J3Y|2/25/2021 |
|LON00600OKDVJML|CUSPZHN9T93TO9S|6/1/2019  |
|LON00826JQ00XJT|CUSOJDLK2VRS15Q|3/14/2024 |
|LON009G20BSWZN3|CUS1EVGVTN77RQD|9/2/2023  |
|LON00AVKGYRPVPC|CUSQWXK8ARUOEWS|2/17/2024 |
|LON00B64UQ7WVU6|CUSJCI07BWW0MOI|3/8/2022  |
|LON00DGAS821QP4|CUSMUA4IQDL6R7N|10/4/2022 |
|LON00EL86S4CSZV|CUSR9B5F6FVALAE|6/15/2022 |
+---------------+---------------+----------+
only showing top 10 rows



In [4]:
from pyspark.sql import functions as F

# Read Silver loans
df_silver = spark.read.table("silver_loans")

# Create Gold fact_loan with correctly converted date
fact_loan = df_silver.select(
    "loan_id",
    "customer_id",
    "loan_amount",
    "interest_rate",
    F.to_date("start_date", "M/d/yyyy").alias("start_date")
)

# Check the result before writing
print("Total loans:", fact_loan.count())
print("Blank start dates:", fact_loan.filter(F.col("start_date").isNull()).count())

fact_loan.select(
    "loan_id",
    "customer_id",
    "loan_amount",
    "interest_rate",
    "start_date"
).show(10, False)

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 6, Finished, Available, Finished, False)

Total loans: 30000
Blank start dates: 0
+---------------+---------------+-----------+-------------+----------+
|loan_id        |customer_id    |loan_amount|interest_rate|start_date|
+---------------+---------------+-----------+-------------+----------+
|LON003ALP7FPVI4|CUSPHZ9KHHK6NAP|193249.96  |5.53         |2022-07-26|
|LON0049M9DPQCZX|CUSDRIYWAS9HTS3|53286.04   |8.59         |2024-06-30|
|LON005JTKPC1NXJ|CUSGUZN0EE60J3Y|264246.73  |13.17        |2021-02-25|
|LON00600OKDVJML|CUSPZHN9T93TO9S|143673.14  |8.52         |2019-06-01|
|LON00826JQ00XJT|CUSOJDLK2VRS15Q|287876.58  |13.62        |2024-03-14|
|LON009G20BSWZN3|CUS1EVGVTN77RQD|79175.1    |12.98        |2023-09-02|
|LON00AVKGYRPVPC|CUSQWXK8ARUOEWS|118263.0   |9.29         |2024-02-17|
|LON00B64UQ7WVU6|CUSJCI07BWW0MOI|176690.06  |13.72        |2022-03-08|
|LON00DGAS821QP4|CUSMUA4IQDL6R7N|110331.81  |2.66         |2022-10-04|
|LON00EL86S4CSZV|CUSR9B5F6FVALAE|152527.16  |5.59         |2022-06-15|
+---------------+---------------+----

In [5]:
fact_loan.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_loan")

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 7, Finished, Available, Finished, False)

In [6]:
df_check = spark.read.table("fact_loan")

print("Total loans:", df_check.count())
print("Blank start dates:", df_check.filter(F.col("start_date").isNull()).count())

df_check.printSchema()

StatementMeta(, 86f2515f-8c85-4b7c-9bda-27e8ffeac447, 8, Finished, Available, Finished, False)

Total loans: 30000
Blank start dates: 0
root
 |-- loan_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- loan_amount: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- start_date: date (nullable = true)



In [5]:
from pyspark.sql import functions as F

df = spark.read.table("silver_accounts")

dim_account = df.select(
    "account_id",
    "customer_id",
    "account_type",
    "balance_usd",
    "open_date"
).withColumn(
    "open_year",
    F.year(F.to_date("open_date"))
)

dim_account.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_account")

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 7, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

df = spark.read.table("silver_accounts")

dim_account = df.select(
    "account_id",
    "customer_id",
    "account_type",
    "balance_usd",
    "open_date"
).withColumn(
    "open_year",
    F.year(F.to_date("open_date"))
)

dim_account.printSchema()

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 4, Finished, Available, Finished, False)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance_usd: double (nullable = true)
 |-- open_date: date (nullable = true)
 |-- open_year: integer (nullable = true)



In [6]:
dim_account.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_account")

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 8, Finished, Available, Finished, False)

In [7]:
spark.read.table("dim_account").printSchema()

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 9, Finished, Available, Finished, False)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance_usd: double (nullable = true)
 |-- open_date: date (nullable = true)
 |-- open_year: integer (nullable = true)



In [8]:
spark.read.table("dim_account").printSchema()

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 10, Finished, Available, Finished, False)

root
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- balance_usd: double (nullable = true)
 |-- open_date: date (nullable = true)
 |-- open_year: integer (nullable = true)



In [9]:
printSchema()

StatementMeta(, bc1608d7-61e5-40ae-a835-900fac8d53e2, 11, Finished, Available, Finished, False)

NameError: name 'printSchema' is not defined

In [1]:
from pyspark.sql import functions as F

# Read existing Gold customer table
df = spark.read.table("dim_customer")

# Create credit score bands
df_banded = df.withColumn(
    "credit_score_band",
    F.when(F.col("credit_score").isNull(), "Unknown")
     .when(F.col("credit_score") < 400, "300-399")
     .when(F.col("credit_score") < 500, "400-499")
     .when(F.col("credit_score") < 600, "500-599")
     .when(F.col("credit_score") < 700, "600-699")
     .when(F.col("credit_score") < 800, "700-799")
     .otherwise("800+")
)

# Check the result
df_banded.groupBy("credit_score_band") \
    .count() \
    .orderBy("credit_score_band") \
    .show()

StatementMeta(, 83586186-a2e6-4c81-9e23-8f725f746756, 3, Finished, Available, Finished, False)

+-----------------+-----+
|credit_score_band|count|
+-----------------+-----+
|          300-399| 9080|
|          400-499| 9078|
|          500-599| 9138|
|          600-699| 9118|
|          700-799| 9051|
|             800+| 4535|
+-----------------+-----+



In [2]:
df_banded.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_customer")

StatementMeta(, 83586186-a2e6-4c81-9e23-8f725f746756, 4, Finished, Available, Finished, False)

In [3]:
check = spark.read.table("dim_customer")

check.select(
    "customer_id",
    "credit_score",
    "credit_score_band"
).show(10, False)

check.printSchema()

StatementMeta(, 83586186-a2e6-4c81-9e23-8f725f746756, 5, Finished, Available, Finished, False)

+---------------+------------+-----------------+
|customer_id    |credit_score|credit_score_band|
+---------------+------------+-----------------+
|CUS000MKX5RHTAP|827         |800+             |
|CUS0079NNF0ML3Z|678         |600-699          |
|CUS00A58YMOM4LW|310         |300-399          |
|CUS00AZDG19RALF|611         |600-699          |
|CUS00GQWY18FJSC|684         |600-699          |
|CUS00I9HUFUMDB5|642         |600-699          |
|CUS00J12MCEOVH5|602         |600-699          |
|CUS013M0U1EX754|687         |600-699          |
|CUS015951MF2S37|802         |800+             |
|CUS01BYWPJNWDUM|752         |700-799          |
+---------------+------------+-----------------+
only showing top 10 rows

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- created_at: string (nullable = tru